# N127 · 质数筛、最小质因子与分解

**目标：** 按查询类型选择试除、筛或预处理。

**先修：** N126, N011。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 试除到平方根；Eratosthenes；线性筛导览；SPF；因子分解；质因数指数。

**配套讲解来源：** [同名逐章意见](../../comment/127_primes_sieves_factorization.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

这一章要解决的问题是：**给你一堆整数，你要快速回答两类问题——"它是不是质数？"和"它到底由哪些质数相乘得到？"** 前者叫素性判断，后者叫质因数分解。

笔记本 “学习目标”部分 里说目标是"按查询类型选择试除、筛或预处理"，翻译成大白话就是：查询只有一两次时，用最笨但最省事的试除法就够了；查询有很多次时，与其每次都从头算，不如先花一次功夫把一张"质数表"或"最小质因子表"造出来，之后每次查询都变成查表，快得多。

给你具体到数字的例子：

- 输入：`x = 60`
- 输出：`{2: 2, 3: 1, 5: 1}`
- 为什么是它？因为 60 = 2 × 2 × 3 × 5，也就是说 60 由两个 2、一个 3、一个 5 连乘得到，代码用字典"质数 → 出现次数"来表示这个答案。

再看造表的例子：输入 `n = 10`，`sieve(10)` 输出 `[2, 3, 5, 7]`，因为小于 10 的整数里只有这四个除了 1 和自身没有别的因子。

## 2. 基础知识：先读懂这些词

- **试除法（trial division）**：判断 x 是不是质数的最直接办法——拿 2、3、4……挨个去试能不能整除 x，全都除不开就是质数。它只需要检查到 √x 就够，因为如果 x 有个大于 √x 的因子，那它必然还配着一个小于 √x 的因子，早就被试出来了。
- **埃拉托斯特尼筛（Eratosthenes sieve，简称筛法）**：反过来想——与其逐个数检查，不如把合数成批划掉。从 2 开始，每遇到一个还没被划掉的数 p，它一定是质数，然后把 p 的倍数 2p、3p、4p……统统划掉。最后没被划掉的就是质数。
- **从 p² 开始标记**：筛法的一个省力技巧。轮到质数 p 时，p² 以下的倍数（比如 2p、3p）早就在处理更小的质数时被划掉了，所以直接从 p² 开始划就行。
- **SPF 表（smallest prime factors，最小质因子表）**：一张下标从 0 到 n 的数组，`spf[x]` 存"x 的最小质因子"（质数 x 的最小质因子就是它自己）。有了这张表，分解任何 x 都不用试除，只要反复查表、除掉最小因子即可。
- **质因数指数**：分解结果里每个质数出现的次数。比如 8 = 2³，指数就是 3。代码里用字典 `{质数: 指数}` 存，比存一串 [2,2,2] 更紧凑。
- **isqrt（整数平方根）**：Python 标准库 `math.isqrt(n)` 返回不超过 √n 的最大整数，比如 `isqrt(10) = 3`。筛法外层循环只需要走到它。

## 3. 思路推导：从小例子开始

- **Step 1：先明确查询模式。** 只判断一个数？用试除。要列出 2 到 n−1 之间所有质数？用筛。要对很多个数做分解？先造 SPF 表再逐个查。
- **Step 2：筛法的核心动作是"划掉倍数"。** 从 p = 2 出发，p 是质数当且仅当它还没被任何更小的质数划掉。
- **Step 3：划倍数时从 p² 起步。** p·2、p·3 这些更小的倍数含有比 p 更小的质因子，处理那个更小质数时就已经划掉了，重划一遍纯属浪费。
- **Step 4：把"判断"升级成"查表"。** 筛法顺手可以多记一层信息：每个合数被划掉时，划它的那个质数 p 就是它的最小质因子。把这件事存进数组，就得到 SPF 表。
- **Step 5：分解变成"不断除以最小因子"。** 要分解 x，就查 `spf[x]` 拿到最小质因子 p，把 p 记进答案、x 除以 p；重复直到 x 变成 1。

**手算演示一：sieve(10) 怎么划掉合数。** 开一个 0 到 9 的标记数组，先划掉 0 和 1。外层 p 走 2 和 3（`isqrt(9) + 1 = 4`，`range(2, 4)` 就是 2、3）：

| 轮次 | p | 动作 | 划掉的数 |
|------|---|------|----------|
| 1 | 2 | 从 2²=4 开始，每隔 2 划一个 | 4、6、8 |
| 2 | 3 | 从 3²=9 开始，每隔 3 划一个 | 9 |

没被划掉的 2、3、5、7 就是答案 `[2, 3, 5, 7]`。注意 3 这轮没划 6（因为 3²=9 才起步），6 早在 p=2 那轮被划掉了——这就是"从 p² 起步"省下的功夫。

**手算演示二：用 SPF 表分解 60。** 查表循环过程如下：

| 轮次 | 当前 x | spf[x] | 记录 | x 除完变成 |
|------|--------|--------|------|-----------|
| 1 | 60 | 2 | {2: 1} | 30 |
| 2 | 30 | 2 | {2: 2} | 15 |
| 3 | 15 | 3 | {2: 2, 3: 1} | 5 |
| 4 | 5 | 5 | {2: 2, 3: 1, 5: 1} | 1 |

x 到 1，循环结束，答案 `{2: 2, 3: 1, 5: 1}`，正好对应 60 = 2² × 3 × 5。

**“运行示例”部分 的表格数据**（notebook 用 `smallest_prime_factors(30)` 造表后展示 2 到 20，摘录前一半）：

| 整数 | 最小质因子 | 质因数分解 |
|------|-----------|-----------|
| 2 | 2 | {2: 1} |
| 3 | 3 | {3: 1} |
| 4 | 2 | {2: 2} |
| 5 | 5 | {5: 1} |
| 6 | 2 | {2: 1, 3: 1} |
| 7 | 7 | {7: 1} |
| 8 | 2 | {2: 3} |
| 9 | 3 | {3: 2} |
| 10 | 2 | {2: 1, 5: 1} |
| 11 | 11 | {11: 1} |

你可以看到：偶数的最小质因子全是 2；9 的最小质因子是 3（不是 9 自己，9 是合数）；质数那一行的最小质因子等于它本身——这正是代码用 `spf[p] == p` 判断"p 是不是质数"的依据。

## 4. 核心代码：sieve

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def sieve(n):
    if n <= 2:
        return []
    prime = [True] * n
    prime[0] = prime[1] = False
    for p in range(2, isqrt(n - 1) + 1):
        if prime[p]:
            for x in range(p * p, n, p):
                prime[x] = False
    return [x for x in range(2, n) if prime[x]]
```

### 逐段读懂代码

“分段实现”部分 一共三个函数，我们逐个看。

**函数一：`sieve(n)` —— 返回所有严格小于 n 的质数。**

```python
from math import isqrt

def sieve(n):
    if n <= 2:
        return []
    prime = [True] * n
    prime[0] = prime[1] = False
    for p in range(2, isqrt(n - 1) + 1):
        if prime[p]:
            for x in range(p * p, n, p):
                prime[x] = False
    return [x for x in range(2, n) if prime[x]]
```

- `if n<=2: return []`：挡住 n 为 0、1、2 的情况。小于 2 的整数里没有任何质数，直接返回空表，免得后面 `[True]*n` 之类越界。
- `prime=[True]*n; prime[0]=prime[1]=False`：一开始假设 0 到 n−1 全是质数，再把 0 和 1 这两个"冒牌货"划掉。这里下标就是数值本身。
- `for p in range(2,isqrt(n-1)+1)`：p 只需要走到 √(n−1)。数组最大下标是 n−1，所以只要 p ≤ √(n−1)，p² 就还在数组范围内；p 再大，p² 越界，标记工作早就做完了。
- `if prime[p]:`：p 还没被划掉，说明 p 是质数，才值得去划它的倍数。已被划掉的 p 是合数，它的倍数会被更小的质数负责。
- `for x in range(p*p,n,p): prime[x]=False`：从 p² 开始、步长 p，把 p 的倍数逐个划掉。这就是第三节演示过的"从 p² 起步"。
- 最后一行列表推导：把仍是 True 的下标收集起来，从 2 起步恰好跳过已划掉的 0 和 1。

**函数二：`smallest_prime_factors(n)` —— 造 0 到 n 的 SPF 表。**

```python
def smallest_prime_factors(n):
    if n < 0:
        raise ValueError('nonnegative bound required')
    spf = list(range(n + 1))
    for p in range(2, isqrt(n) + 1):
        if spf[p] == p:
            for x in range(p * p, n + 1, p):
                if spf[x] == x:
                    spf[x] = p
    return spf
```

- `if n<0: raise ValueError(...)`：挡住负数上界。负数没法当数组长度，直接报错比静默返回怪东西更诚实。
- `spf=list(range(n+1))`：初始时令 `spf[x] = x`，即"每个数暂时认为最小质因子是自己"。质数会永远保持这个值；合数之后会被覆盖。
- `if spf[p]==p:`：这句在问"p 还没被改过吗？"等价于"p 是质数吗？"——只有质数不会被更小的质数划掉。
- `if spf[x]==x: spf[x]=p`：只覆盖还没被覆盖的格子。因为外层 p 从小到大遍历，一个合数第一次被覆盖时用的就是它最小的质因子；之后再遇到更大的质数想覆盖它，条件不成立，不会覆盖——这就是"记录的是最小质因子"的保障。
- 注意这张表长度是 n+1，**包含上界 n**，和 `sieve` 的"严格小于 n"是两个约定，别混。

**函数三：`factorize(x, spf)` —— 查表分解 x，返回 {质数: 指数}。**

```python
def factorize(x, spf):
    if not 1 <= x < len(spf):
        raise ValueError('x must be positive and covered by the SPF table')
    factors = {}
    while x > 1:
        p = spf[x]
        factors[p] = factors.get(p, 0) + 1
        x //= p
    return factors
```

- `if not 1<=x<len(spf): raise ValueError(...)`：挡住两类坏输入——x 小于 1（没法分解）和 x 超出 SPF 表覆盖范围（查表会拿到错误答案）。`1<=x<len(spf)` 连起来读就是"1 ≤ x ≤ 表长−1"，即 x 在表内且有意义。
- `while x>1:`：x 还是大于 1，就说明还没分解完，继续挖因子。x 到 1 自然停。
- `p=spf[x]`：查表拿到当前 x 的最小质因子，一步到位，不用试除。
- `factors[p]=factors.get(p,0)+1`：把 p 的计数加一。`get(p,0)` 的意思是"字典里有 p 就取现值，没有就当 0"，这样同一个质数反复出现（比如 60 里的两个 2）会累加成指数 2。
- `x//=p`：x 整除掉 p，进入下一轮。演示见第三节分解 60 的那张四行表格。

## 5. 分段实现：按顺序运行

**本章接口：** `sieve(n)`、`smallest_prime_factors(n)`、`factorize(x, spf)`

### 导入本章使用的库

In [1]:
from math import isqrt

### sieve

In [2]:
def sieve(n):
    if n <= 2:
        return []
    prime = [True] * n
    prime[0] = prime[1] = False
    for p in range(2, isqrt(n - 1) + 1):
        if prime[p]:
            for x in range(p * p, n, p):
                prime[x] = False
    return [x for x in range(2, n) if prime[x]]

### smallest_prime_factors

In [3]:
def smallest_prime_factors(n):
    if n < 0:
        raise ValueError('nonnegative bound required')
    spf = list(range(n + 1))
    for p in range(2, isqrt(n) + 1):
        if spf[p] == p:
            for x in range(p * p, n + 1, p):
                if spf[x] == x:
                    spf[x] = p
    return spf

### factorize

In [4]:
def factorize(x, spf):
    if not 1 <= x < len(spf):
        raise ValueError('x must be positive and covered by the SPF table')
    factors = {}
    while x > 1:
        p = spf[x]
        factors[p] = factors.get(p, 0) + 1
        x //= p
    return factors

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [5]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

spf=smallest_prime_factors(30)
show_table(['整数','最小质因子','质因数分解'],[(x,spf[x],factorize(x,spf)) for x in range(2,21)])

整数,最小质因子,质因数分解
2,2,{2: 1}
3,3,{3: 1}
4,2,{2: 2}
5,5,{5: 1}
6,2,"{2: 1, 3: 1}"
7,7,{7: 1}
8,2,{2: 3}
9,3,{3: 2}
10,2,"{2: 1, 5: 1}"
11,11,{11: 1}


## 7. 正确性、适用条件与复杂度

每个合数都有不超过平方根的质因子，所以最终都会被标记；质数不会被更小质数整除。SPF只在尚未被标记时赋值，按p从小到大处理保证记录最小质因子。分解每轮严格减小x并保持“已取因子乘积×剩余x=原值”。

**代价：** 埃拉托斯特尼筛O(n log log n)算术工作量、O(n)空间；SPF同级预处理，分解最多O(log x)次整除。sieve返回严格小于n的质数，SPF表包含上界n，两个边界不混用。

### 展开理解

**为什么每个合数都会被筛掉？** 一个合数 x 至少有一个因子不超过 √x（如果它的因子全都大于 √x，两个因子相乘就超过 x 了，矛盾）。质因子也是因子，所以每个合数都有一个不超过 √x 的质因子 p；只要 n 范围内这个 p 走到，p² ≤ x 就会被标记，x 必然被划掉，漏不掉。

**为什么真正的质数不会被误划？** 我们只划"某个质数 p 的倍数"。质数 q 只能被 1 和 q 整除，任何 p < q 的 p 都整不开 q，所以 q 永远不会被划掉。

**为什么 SPF 表存的一定是最小质因子？** 外层 p 严格从小到大走。合数 x 第一次被覆盖，发生在最小的那个能整除 x 的质数身上，此时写入的 p 就是 x 的最小质因子；之后的质数想写它，会先被 `if spf[x]==x` 拦住。

**为什么分解一定结束且不丢东西？** 每轮循环 x 都被除以一个大于 1 的质数，严格变小，所以循环必然终止；同时"已收集因子的乘积 × 当前 x"每一轮都等于原来的 x（只是把因子从 x 挪进了字典），这个平衡始终成立，最后 x=1 时字典里的因子乘起来恰好是原值。

**复杂度（结合具体数字）：** 埃拉托斯特尼筛的总工作量是 O(n log log n)，空间 O(n)。n = 10⁶ 时 log log n 只是 3 左右，总共约三百万次上下的标记操作，一眨眼就算完；n = 10⁷ 也只要几秒级。SPF 表同级。而分解单个 x 最多 O(log x) 次整除——x = 10⁹ 也就 30 来轮（因为每次至少除以 2，翻倍速降），单次查询几乎不要时间。对比一下：如果不用表、每个数都从 2 试除到 √x，x = 10⁹ 就要试三万次，查询一多就顶不住了。

## 8. 单元测试：每个用例在检查什么

“自动测试”部分 的断言分三组：

1. `assert sieve(10)==[2,3,5,7] and sieve(2)==[] and sieve(0)==[]` —— 前半段是**正常值**抽查：小于 10 的质数恰好是 2、3、5、7。后两段是**边界值**：n=2 和 n=0 检查"严格小于 n"这个约定——2 本身是质数但不出现在结果里，返回空表是对的。
2. 1 到 500 的循环检查（配一个暴力试除的 `trial_prime` 当"标准答案"）：
   - `assert trial_prime(p)`：分解出的每个因子 p 确实是质数——防止把 4、6 这种假因子混进答案；
   - `product *= p**e` 后 `assert product==x`：把因子按指数乘回去，必须等于原数——证明分解"不多不少"；
   - `assert spf[x]==next(d for d in range(2,x+1) if x%d==0)`：对每个 x≥2，直接从 2 往上找第一个能整除 x 的数，和 SPF 表对照——证明表存的真是"最小"因子。
3. `assert sieve(501)==[x for x in range(501) if trial_prime(x)]` —— 用暴力法逐个判断 0 到 500 的素性，和筛法结果整表对照，两边完全一致。

这套测试的思路是"拿一个笨但显然正确的暴力实现当裁判"，这也是 “思路推导”部分 里"逐数试除"的用武之地。

In [6]:
assert sieve(10) == [2, 3, 5, 7] and sieve(2) == [] and (sieve(0) == [])

spf = smallest_prime_factors(500)

def trial_prime(x):
    return x >= 2 and all((x % d for d in range(2, isqrt(x) + 1)))

for x in range(1, 501):
    factors = factorize(x, spf)
    product = 1
    for p, e in factors.items():
        assert trial_prime(p)
        product *= p ** e
    assert product == x
    if x >= 2:
        assert spf[x] == next((d for d in range(2, x + 1) if x % d == 0))

assert sieve(501) == [x for x in range(501) if trial_prime(x)]

print('N127: 所有本章断言通过')

N127: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 解释为什么从p²开始筛。

**练习 2：** 批量分解与逐次试除比较。

<details>
<summary>展开思路提示（不是完整答案）</summary>

**练习 1（为什么从 p² 开始筛）：** 提示——设 k < p，考虑倍数 k·p：k 自己必然含有一个小于等于 k 的质因子 q，而 q < p，所以 k·p 在处理 q 的那一轮就已经按 q 的倍数被划掉了。你要做的是把这个论证写完整，并配一个具体例子（比如 p=5 时，10、15、20 分别在谁的轮次被划掉），再用 `sieve` 或手写暴力对照验证。

**练习 2（批量分解 vs 逐次试除）：** 提示——算两笔账。逐次试除：每次分解 x 花 O(√x)，做 m 次就是 m·√x；SPF 方案：一次性 O(n) 造表，之后每次分解只要 O(log x)，总计 n + m·log x。取一组具体数字（比如 n = 10⁵、m = 10⁵、x ≈ 10⁵）代入两边比较，结论就很直观：查询次数一多，预处理稳赢；只查一两次，预处理反而亏。写清楚输入输出和边界（x 超出表覆盖时怎么办），再跑数验证。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [7]:
# 完整实现：本单元格不依赖其他单元格。
from math import isqrt

def sieve(n):
    if n <= 2:
        return []
    prime = [True] * n
    prime[0] = prime[1] = False
    for p in range(2, isqrt(n - 1) + 1):
        if prime[p]:
            for x in range(p * p, n, p):
                prime[x] = False
    return [x for x in range(2, n) if prime[x]]

def smallest_prime_factors(n):
    if n < 0:
        raise ValueError('nonnegative bound required')
    spf = list(range(n + 1))
    for p in range(2, isqrt(n) + 1):
        if spf[p] == p:
            for x in range(p * p, n + 1, p):
                if spf[x] == x:
                    spf[x] = p
    return spf

def factorize(x, spf):
    if not 1 <= x < len(spf):
        raise ValueError('x must be positive and covered by the SPF table')
    factors = {}
    while x > 1:
        p = spf[x]
        factors[p] = factors.get(p, 0) + 1
        x //= p
    return factors

# 示例与回归测试
assert sieve(10) == [2, 3, 5, 7] and sieve(2) == [] and (sieve(0) == [])

spf = smallest_prime_factors(500)

def trial_prime(x):
    return x >= 2 and all((x % d for d in range(2, isqrt(x) + 1)))

for x in range(1, 501):
    factors = factorize(x, spf)
    product = 1
    for p, e in factors.items():
        assert trial_prime(p)
        product *= p ** e
    assert product == x
    if x >= 2:
        assert spf[x] == next((d for d in range(2, x + 1) if x % d == 0))

assert sieve(501) == [x for x in range(501) if trial_prime(x)]

print('N127: 所有本章断言通过')

N127: 所有本章断言通过


## 11. 代表题与迁移

- **204. Count Primes（计数质数）**：给 n 数小于 n 的质数个数——直接练本章的 `sieve`，数一数返回列表的长度即可，是筛法的教科书题。
- **2523. Closest Prime Numbers in Range（范围内最近的两个质数）**：在区间 [left, right] 里找相邻质数差最小的一对——练"筛出一段范围内的质数后再做线性扫描"，考察你把筛法和区间查询组合起来。
- **2507. Smallest Value After Replacing With Sum of Prime Factors（最小质因数替换后的最小值）**：反复把一个数替换成"它所有质因数之和"直到不变——练的就是 `factorize` 拿到质因数及指数（指数就是"同一个质数要加几次"），再把流程迭代到收敛。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。